In [9]:
import pandas as pd

df = pd.read_csv("dataset/synthetic_logs.csv")
df

df[df.target_label == 'System Notification'].sample(10)


,timestamp,source,log_message,target_label,complexity
1172,12/15/2025 20:59,ModernHR,File data_2843.csv uploaded successfully by us...,System Notification,regex
185,3/11/2025 21:44,ModernCRM,System updated to version 4.7.4.,System Notification,regex
2018,11/10/2025 15:07,ModernCRM,File data_2061.csv uploaded successfully by us...,System Notification,regex
1878,5/27/2025 22:11,BillingSystem,Disk cleanup completed successfully.,System Notification,regex
1778,6/15/2025 6:02,ThirdPartyAPI,Backup started at 2025-11-06 10:01:01.,System Notification,regex
552,9/22/2025 20:54,ModernHR,System reboot initiated by user User421.,System Notification,regex
1134,1/19/2025 9:13,ThirdPartyAPI,Backup ended at 2025-01-29 23:05:36.,System Notification,regex
1798,6/21/2025 6:48,BillingSystem,Backup completed successfully.,System Notification,regex
972,3/21/2025 23:04,AnalyticsEngine,File data_1124.csv uploaded successfully by us...,System Notification,regex
325,6/2/2025 12:18,ThirdPartyAPI,Backup ended at 2025-07-18 17:06:54.,System Notification,regex


In [10]:
df.source.unique()


<StringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [11]:
df.target_label.unique()


<StringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [16]:
print(df[df.target_label=='System Notification'].sample(10).to_string())

            timestamp           source                                                log_message         target_label complexity
2246   3/2/2025 22:56         ModernHR                   System reboot initiated by user User488.  System Notification      regex
913    7/26/2025 2:50    ThirdPartyAPI                           System updated to version 4.6.3.  System Notification      regex
1555  9/10/2025 22:19    BillingSystem                     Backup started at 2025-03-04 20:49:02.  System Notification      regex
1856   6/27/2025 4:56         ModernHR  File data_1714.csv uploaded successfully by user User967.  System Notification      regex
741    2/12/2025 8:21  AnalyticsEngine  File data_4085.csv uploaded successfully by user User222.  System Notification      regex
244     7/5/2025 2:52        ModernCRM  File data_2127.csv uploaded successfully by user User577.  System Notification      regex
1651   6/2/2025 23:20        ModernCRM                             Backup completed succes

In [13]:
print(df[df.log_message.str.startswith("System reboot initiated by user")])

             timestamp           source  \
36    11/19/2025 13:14    BillingSystem   
92     12/4/2025 21:20    BillingSystem   
139     5/8/2025 16:34         ModernHR   
140     9/11/2025 8:49  AnalyticsEngine   
161    3/31/2025 19:40    BillingSystem   
163     6/6/2025 15:29    BillingSystem   
307     4/12/2025 0:41    BillingSystem   
365   10/20/2025 22:32         ModernHR   
508     4/15/2025 2:04    ThirdPartyAPI   
552    9/22/2025 20:54         ModernHR   
668      9/5/2025 7:14         ModernHR   
693     7/6/2025 21:40    BillingSystem   
697     3/13/2025 7:09    BillingSystem   
714    9/25/2025 23:35    ThirdPartyAPI   
730    5/24/2025 11:08  AnalyticsEngine   
800    8/15/2025 12:14    BillingSystem   
837      4/9/2025 8:28  AnalyticsEngine   
852     3/31/2025 5:20        ModernCRM   
865     2/25/2025 1:40  AnalyticsEngine   
889   11/30/2025 13:45         ModernHR   
896    7/28/2025 11:24    BillingSystem   
988    9/11/2025 22:23    BillingSystem   
1106  12/28

In [19]:
from sklearn.cluster import DBSCAN
from sentence_transformers import SentenceTransformer

C:\code\Classification-log\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight embedding model
embeddings = model.encode(df['log_message'].tolist())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4584.33it/s]


In [21]:
embeddings[:5]


array([[-0.10293962,  0.03354597, -0.02202607, ...,  0.00457792,
        -0.04259717,  0.00322622],
       [ 0.00804573, -0.03573923,  0.04938737, ...,  0.01538319,
        -0.06230948, -0.02774663],
       [-0.00908223,  0.13003924, -0.05275566, ...,  0.02014104,
        -0.05117098, -0.02930296],
       [-0.09751045,  0.04911297, -0.03977424, ...,  0.02477499,
        -0.03546081, -0.00018602],
       [-0.10468341,  0.05926036, -0.02488498, ...,  0.02502053,
        -0.03719297, -0.02568912]], shape=(5, 384), dtype=float32)

In [22]:
clustering = DBSCAN(eps=0.2, min_samples=1, metric='cosine').fit(embeddings)
df['cluster'] = clustering.labels_

In [24]:
print(df.head())

             timestamp           source  \
0  2025-06-27 07:20:25        ModernCRM   
1      1/14/2025 23:07        ModernCRM   
2       1/17/2025 1:29  AnalyticsEngine   
3  2025-07-12 00:24:16         ModernHR   
4  2025-06-02 18:25:23    BillingSystem   

                                         log_message    target_label  \
0  nova.osapi_compute.wsgi.server [req-b9718cd8-f...     HTTP Status   
1     Email service experiencing issues with sending  Critical Error   
2          Unauthorized access to data was attempted  Security Alert   
3  nova.osapi_compute.wsgi.server [req-4895c258-b...     HTTP Status   
4  nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...     HTTP Status   

  complexity  cluster  
0       bert        0  
1       bert        1  
2       bert        2  
3       bert        0  
4       bert        0  


In [25]:
# Group by cluster to inspect patterns
clusters = df.groupby('cluster')['log_message'].apply(list)
sorted_clusters = clusters.sort_values(key=lambda x: x.map(len), ascending=False)

In [26]:
print("Clustered Patterns:")
for cluster_id, messages in sorted_clusters.items():
    if len(messages) > 10:
        print(f"Cluster {cluster_id}:")
        for msg in messages[:5]:
            print(f"  {msg}")

Clustered Patterns:
Cluster 0:
  nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2675118
  nova.osapi_compute.wsgi.server [req-4895c258-b2f8-488f-a2a3-4fae63982e48 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" HTTP status code -  200 len: 211 time: 0.0968180
  nova.osapi_compute.wsgi.server [req-ee8bc8ba-9265-4280-9215-dbe000a41209 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" RCODE  200 len: 1874 time: 0.2280791
  nova.osapi_compute.wsgi.server [req-f0bffbc3-5ab0-4916-91c1-0a61dd7d4ec2 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2

In [27]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message):
            return label
    return None

In [28]:
classify_with_regex("User User123 logged in.")

'User Action'

In [29]:
classify_with_regex("System reboot initiated by user User179.")

'System Notification'

In [30]:
classify_with_regex("Hey you, chill bro")

In [35]:
# Apply regex classification
df['regex_label'] = df['log_message'].apply(lambda x: classify_with_regex(x))
print(df[df['regex_label'].notnull()].to_string())

             timestamp           source                                                log_message         target_label complexity  cluster          regex_label
7      10/11/2025 8:44         ModernHR  File data_6169.csv uploaded successfully by user User953.  System Notification      regex        4  System Notification
14       1/4/2025 1:43    ThirdPartyAPI  File data_3847.csv uploaded successfully by user User175.  System Notification      regex        4  System Notification
15       5/1/2025 9:41        ModernCRM                             Backup completed successfully.  System Notification      regex        8  System Notification
18     2/22/2025 17:49        ModernCRM                   Account with ID 5351 created by User634.          User Action      regex        9          User Action
27     9/24/2025 19:57    ThirdPartyAPI                                   User User685 logged out.          User Action      regex       11          User Action
30      4/26/2025 7:54  AnalyticsE

In [37]:
print(df[df['regex_label'].isnull()].head(5))

             timestamp           source  \
0  2025-06-27 07:20:25        ModernCRM   
1      1/14/2025 23:07        ModernCRM   
2       1/17/2025 1:29  AnalyticsEngine   
3  2025-07-12 00:24:16         ModernHR   
4  2025-06-02 18:25:23    BillingSystem   

                                         log_message    target_label  \
0  nova.osapi_compute.wsgi.server [req-b9718cd8-f...     HTTP Status   
1     Email service experiencing issues with sending  Critical Error   
2          Unauthorized access to data was attempted  Security Alert   
3  nova.osapi_compute.wsgi.server [req-4895c258-b...     HTTP Status   
4  nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...     HTTP Status   

  complexity  cluster regex_label  
0       bert        0         NaN  
1       bert        1         NaN  
2       bert        2         NaN  
3       bert        0         NaN  
4       bert        0         NaN  


In [38]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex.shape

(1910, 7)

In [40]:
df_legacy = df_non_regex[df_non_regex.source=="LegacyCRM"]
print(df_legacy)

                timestamp     source  \
60    2025-10-06 16:55:23  LegacyCRM   
255   2025-05-03 16:55:35  LegacyCRM   
377   2025-06-24 12:16:29  LegacyCRM   
1325  2025-04-17 07:33:44  LegacyCRM   
1734  2025-04-30 07:47:30  LegacyCRM   
1826  2025-01-23 10:33:36  LegacyCRM   
2217  2025-05-12 09:46:54  LegacyCRM   

                                            log_message         target_label  \
60    Lead conversion failed for prospect ID 7842 du...       Workflow Error   
255   API endpoint 'getCustomerDetails' is deprecate...  Deprecation Warning   
377   Customer follow-up process for lead ID 5621 fa...       Workflow Error   
1325  Escalation rule execution failed for ticket ID...       Workflow Error   
1734  The 'ExportToCSV' feature is outdated. Please ...  Deprecation Warning   
1826  Support for legacy authentication methods will...  Deprecation Warning   
2217  Task assignment for TeamID 3425 could not comp...       Workflow Error   

     complexity  cluster regex_label  

In [41]:
df_non_legacy = df_non_regex[df_non_regex.source!="LegacyCRM"]
print(df_non_legacy)

                timestamp           source  \
0     2025-06-27 07:20:25        ModernCRM   
1         1/14/2025 23:07        ModernCRM   
2          1/17/2025 1:29  AnalyticsEngine   
3     2025-07-12 00:24:16         ModernHR   
4     2025-06-02 18:25:23    BillingSystem   
...                   ...              ...   
2405  2025-08-13 07:29:25         ModernHR   
2406       1/11/2025 5:32         ModernHR   
2407  2025-08-03 03:07:47    ThirdPartyAPI   
2408     11/11/2025 11:52    BillingSystem   
2409     12/25/2025 13:21  AnalyticsEngine   

                                            log_message    target_label  \
0     nova.osapi_compute.wsgi.server [req-b9718cd8-f...     HTTP Status   
1        Email service experiencing issues with sending  Critical Error   
2             Unauthorized access to data was attempted  Security Alert   
3     nova.osapi_compute.wsgi.server [req-4895c258-b...     HTTP Status   
4     nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...     HTTP Status 

In [42]:
df_non_legacy.shape

(1903, 7)

In [43]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight embedding model
embeddings_filtered = model.encode(df_non_legacy['log_message'].tolist())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7488.14it/s]


In [44]:
len(embeddings_filtered)

1903

In [45]:
X = embeddings_filtered
y = df_non_legacy['target_label'].values

In [46]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



In [50]:
import os
os.makedirs('../models', exist_ok=True)

In [51]:
import joblib
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(clf, '../models/log_classifier.joblib')
print("model saved successfully!")

model saved successfully!
